In [ ]:
import mdtraj as md
import numpy as np
import pandas as pd
import pickle
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
from pathlib import Path

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

from sklearn.feature_selection import VarianceThreshold
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.linear_model import LogisticRegression

sys.path.insert(0, '/home/rzhu/Desktop/projects/kinase_analysis/src/')
from utils import *


In this notebook I compare the dominant different features between microstates in following steps:
1. Compute feature including C-a pairwise distances, dihedral angles, and sidechain rotamers. 
2. Initial screening: t-test on features 
3. Train a (linear?) classfier


In [ ]:
# Define cache file paths
cache_dir = Path('cache_abl_features')
cache_dir.mkdir(exist_ok=True)

s1_features_file = cache_dir / 's1_features.npy'
s2_features_file = cache_dir / 's2_features.npy'
feature_info_file = cache_dir / 'feature_info.pkl'
atom_indices_info_file = cache_dir / 'atom_indices_info.pkl'
selected_idx_file = cache_dir / 'selected_idx.npy'

features_cached = all([
    s1_features_file.exists(),
    s2_features_file.exists(), 
    feature_info_file.exists(),
    atom_indices_info_file.exists()
])

processed_features_cached = features_cached and selected_idx_file.exists()

print(f"Raw features cached: {features_cached}")
print(f"Processed features cached: {processed_features_cached}")

if processed_features_cached:
    print("Loading processed features and selected indices from cache...")
    s1_features = np.load(s1_features_file)
    s2_features = np.load(s2_features_file)
    selected_idx = np.load(selected_idx_file)
    
    with open(feature_info_file, 'rb') as f:
        feature_info = pickle.load(f)
    
    with open(atom_indices_info_file, 'rb') as f:
        atom_indices_info = pickle.load(f)
    
    # Load minimal trajectory for feature interpretation
    s1_traj_dir = Path('/home/rzhu/Desktop/projects/kinase_analysis/data/abl/msm/scan_lags/11/samples/samples_bottleneck')
    s1_traj = [md.load(f) for f in [list(s1_traj_dir.glob('*.pdb'))[0]]]
    
    print(f"Loaded s1_features: {s1_features.shape}")
    print(f"Loaded s2_features: {s2_features.shape}")
    print(f"Loaded selected_idx: {len(selected_idx)} features")
    print(f"Feature info keys: {list(feature_info.keys())}")
    
    # Create processed X and y directly
    X = np.concatenate([s1_features, s2_features], axis=0)[:, selected_idx]
    y = np.array([0]*len(s1_features) + [1]*len(s2_features))
    print(f"Created X: {X.shape}, y: {y.shape}")
    
elif features_cached:
    print("Loading raw features from cache (will need to compute selected_idx)...")
    s1_features = np.load(s1_features_file)
    s2_features = np.load(s2_features_file)
    
    with open(feature_info_file, 'rb') as f:
        feature_info = pickle.load(f)
    
    with open(atom_indices_info_file, 'rb') as f:
        atom_indices_info = pickle.load(f)
    
    # Load minimal trajectory for feature interpretation  
    s1_traj_dir = Path('/home/rzhu/Desktop/projects/kinase_analysis/data/abl/msm/scan_lags/11/samples/samples_bottleneck')
    s1_traj = [md.load(f) for f in [list(s1_traj_dir.glob('*.pdb'))[0]]]
    
    print(f"Loaded s1_features: {s1_features.shape}")
    print(f"Loaded s2_features: {s2_features.shape}")
    print(f"Feature info keys: {list(feature_info.keys())}")
else:
    print("No cached features found. Will compute features from trajectories.")

In [ ]:
# Compute or load features
if not features_cached:
    # Load trajectories
    # s1_traj_dir = Path('/home/rzhu/Desktop/projects/kinase_analysis/data/abl/msm/scan_lags/11/samples/samples_bottleneck')
    # s2_traj_dir = Path('/home/rzhu/Desktop/projects/kinase_analysis/data/abl/msm/scan_lags/11/samples/post_bottleneck_state')
    
    # s1_traj = [md.load(f) for f in s1_traj_dir.glob('*.pdb')]
    # s2_traj = [md.load(f) for f in s2_traj_dir.glob('*.pdb')]
    s1_traj = [md.load('egfr_smamples/bottleneck_1000_samples.pdb')]
    s2_traj = [md.load('egfr_smamples/post_bottleneck_1000_samples.pdb')]

    # Featurize trajectories
    print(f"Computing features for {len(s1_traj)} + {len(s2_traj)} trajectories...")
    s1_features, feature_info, atom_indices_info = featurize_trajectories(s1_traj)
    s2_features, _, _ = featurize_trajectories(s2_traj)
    
    # Cache results
    np.save(s1_features_file, s1_features)
    np.save(s2_features_file, s2_features)
    with open(feature_info_file, 'wb') as f:
        pickle.dump(feature_info, f)
    with open(atom_indices_info_file, 'wb') as f:
        pickle.dump(atom_indices_info, f)
    
    print(f"Features computed and cached: {s1_features.shape[1]} dimensions")
else:
    print(f"Features loaded from cache: {s1_features.shape[1]} dimensions")

In [ ]:
# Feature summary
print(f"Raw features: S1{s1_features.shape}, S2{s2_features.shape}")
print(f"Feature types: {', '.join(f'{k}({v[1]-v[0]})' for k, v in feature_info.items())}")

In [ ]:
# Feature selection and data preparation
if not processed_features_cached:
    # Apply feature cleanup
    selected_idx = feature_cleanup_with_stats(
        s1_features, s2_features,
        var_threshold=1e-3,
        corr_threshold=0.85
    )
    np.save(selected_idx_file, selected_idx)  # Cache for next time

# Create final ML dataset
X = np.concatenate([s1_features, s2_features], axis=0)[:, selected_idx]
y = np.array([0]*len(s1_features) + [1]*len(s2_features))

# Summary
print(f"Features: {s1_features.shape[1]} → {len(selected_idx)} → Ready for ML")
print(f"Dataset: X{X.shape}, y{y.shape} | S1: {np.sum(y==0)}, S2: {np.sum(y==1)}")

Logistic regression

---

In [ ]:
pipe = Pipeline([
    # scale only within train folds; many distance/angle features benefit from z-scoring
    ("scaler", StandardScaler(with_mean=True)),
    # per-fold univariate shrinkage
    ("anova", SelectKBest(score_func=f_classif, k=1000)),  # k will be tuned
    # per-fold embedded sparse selection
    ("sparse", SelectFromModel(
        LogisticRegression(penalty="l1", solver="saga", max_iter=4000, n_jobs=-1),
        max_features=None, threshold="median"  # threshold will be overridden via estimator C
    )),
    # final interpretable classifier
    ("clf", LogisticRegression(penalty="l2", solver="lbfgs", max_iter=4000))
])

param_grid = {
    # how aggressively to pre-shrink with ANOVA
    "anova__k": [300, 500, 800, 1000],
    # sparsity strength in the selector (bigger C = less shrinkage)
    "sparse__estimator__C": [0.05, 0.1, 0.2, 0.5, 1.0],
    # final classifier regularization
    "clf__C": [0.5, 1.0, 2.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
gs = GridSearchCV(pipe, param_grid=param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
gs.fit(X, y)

print("Best CV AUC:", gs.best_score_)
print("Best params:", gs.best_params_)

In [ ]:
best_pipe = gs.best_estimator_   # retrieve the full pipeline
anova  = best_pipe.named_steps["anova"]
sparse = best_pipe.named_steps["sparse"]
clf    = best_pipe.named_steps["clf"]

# Get ANOVA-selected features
mask_anova = anova.get_support()  # boolean mask for ANOVA selection
selected_idx_anova = selected_idx[mask_anova]  # map back to original indices

# Get sparse-selected features from ANOVA set
mask_sparse = sparse.get_support()  # boolean mask for sparse selection  
selected_idx_final = selected_idx_anova[mask_sparse]  # final feature indices

In [ ]:
# Write feature interpretation to file
output_file = Path("abl_logistic_regression_features.txt")

with open(output_file, 'w') as f:
    f.write("Final selected features with their structural meaning:\n")
    f.write("=" * 60 + "\n\n")
    
    # Get coefficients and pair with final feature indices
    coef = clf.coef_[0]
    feature_importance = list(zip(selected_idx_final, coef))
    
    # Sort by absolute coefficient value (most important features)
    feature_importance.sort(key=lambda x: abs(x[1]), reverse=True)
    
    f.write(f"All {len(feature_importance)} features ranked by importance:\n")
    f.write("-" * 50 + "\n")
    
    for i, (original_idx, coefficient) in enumerate(feature_importance):
        ftype, atoms_desc, atom_idx = get_feature_atoms(original_idx, feature_info, atom_indices_info, s1_traj[0])
        direction = "favors S2" if coefficient > 0 else "favors S1"
        f.write(f"{i+1:3d}. Feature {original_idx:4d} (coef={coefficient:7.4f}, {direction:11s}): {ftype:12s} - {atoms_desc}\n")
    
    f.write(f"\n" + "=" * 60 + "\n")
    f.write(f"SUMMARY STATISTICS\n")
    f.write(f"=" * 60 + "\n")
    f.write(f"Total features in final model: {len(selected_idx_final)}\n")
    f.write(f"Features favoring S1 (bottleneck): {sum(1 for c in coef if c < 0)}\n")
    f.write(f"Features favoring S2 (post-bottleneck): {sum(1 for c in coef if c > 0)}\n\n")
    
    f.write(f"Feature selection summary:\n")
    f.write(f"Original features: {s1_features.shape[1]}\n")
    f.write(f"After variance/correlation cleanup: {len(selected_idx)}\n")
    f.write(f"After ANOVA selection: {len(selected_idx_anova)}\n")
    f.write(f"After sparse selection (final): {len(selected_idx_final)}\n")
    reduction_pct = 100 * (1 - len(selected_idx_final) / s1_features.shape[1])
    f.write(f"Total reduction: {reduction_pct:.1f}%\n\n")
    
    # Add feature type breakdown
    f.write(f"FEATURE TYPE BREAKDOWN\n")
    f.write(f"=" * 60 + "\n")
    feature_types = {}
    for original_idx, coefficient in feature_importance:
        ftype, atoms_desc, atom_idx = get_feature_atoms(original_idx, feature_info, atom_indices_info, s1_traj[0])
        if ftype not in feature_types:
            feature_types[ftype] = []
        feature_types[ftype].append((original_idx, coefficient, atoms_desc))
    
    for ftype, features in feature_types.items():
        f.write(f"\n{ftype.upper()} ({len(features)} features):\n")
        f.write("-" * 30 + "\n")
        for original_idx, coefficient, atoms_desc in features[:10]:  # Show top 10 per type
            direction = "favors S2" if coefficient > 0 else "favors S1"
            f.write(f"  Feature {original_idx:4d} (coef={coefficient:7.4f}, {direction:11s}): {atoms_desc}\n")
        if len(features) > 10:
            f.write(f"  ... and {len(features) - 10} more\n")

print(f"Feature interpretation written to: {output_file.absolute()}")
print(f"File contains {len(feature_importance)} ranked features with structural annotations")

In [ ]:
coefficient_matrix = np.zeros([len(list(s1_traj[0].topology.residues)), len(list(s1_traj[0].topology.residues))])
first_res_id = int(str(list(s1_traj[0].topology.residues)[0])[3:])
for i, (original_idx, coefficient) in enumerate(feature_importance):
    ftype, atoms_desc, atom_idx = get_feature_atoms(original_idx, feature_info, atom_indices_info, s1_traj[0])
    if 'ca_distances' in ftype:
        atom_id_0 = int(atoms_desc.split(' - ')[0].split()[1][3:]) - first_res_id
        atom_id_1 = int(atoms_desc.split(' - ')[1].split()[1][3:]) - first_res_id

        coefficient_matrix[atom_id_0, atom_id_1] = coefficient
        coefficient_matrix[atom_id_1, atom_id_0] = coefficient

In [ ]:
# Plot coefficient matrix as a 2D heatmap
fig, ax = plt.subplots(figsize=(12, 10))

# Create heatmap with diverging blue-white-red colormap centered at 0
# Calculate symmetric range around 0 for proper centering
vmax = max(abs(coefficient_matrix.min()), abs(coefficient_matrix.max()))
im = ax.imshow(coefficient_matrix, cmap='bwr', aspect='auto', interpolation='nearest', 
               vmin=-vmax, vmax=vmax)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('Logistic Regression Coefficient', rotation=270, labelpad=20)

# Set labels and title
ax.set_xlabel('Residue Index')
ax.set_ylabel('Residue Index')
ax.set_title('Coefficient Matrix: CA Distance Features\n(Blue: favors S1/bottleneck, Red: favors S2/post-bottleneck)')

# Add grid for better readability
ax.grid(True, alpha=0.3)

# Set ticks at regular intervals
n_residues = coefficient_matrix.shape[0]
tick_interval = max(1, n_residues // 20)  # Show ~20 ticks
ticks = range(0, n_residues, tick_interval)
ax.set_xticks(ticks)
ax.set_yticks(ticks)

# Add residue numbers (accounting for first_res_id offset)
tick_labels = [str(first_res_id + t) for t in ticks]
ax.set_xticklabels(tick_labels)
ax.set_yticklabels(tick_labels)

plt.tight_layout()
plt.savefig('abl_logistic_regression_importance.png', dpi=300)
plt.show()

# Print some statistics about the coefficient matrix
print(f"Coefficient matrix shape: {coefficient_matrix.shape}")
print(f"Non-zero elements: {np.count_nonzero(coefficient_matrix)}")
print(f"Min coefficient: {coefficient_matrix.min():.4f}")
print(f"Max coefficient: {coefficient_matrix.max():.4f}")
print(f"Range: {coefficient_matrix.max() - coefficient_matrix.min():.4f}")
print(f"First residue ID: {first_res_id}")
print(f"Last residue ID: {first_res_id + coefficient_matrix.shape[0] - 1}")

In [ ]:
# Plot only the dihedral features that were actually selected by the logistic regression model
print(f"Analyzing dihedral features from the {len(selected_idx_final)} features selected by logistic regression...")

# Collect dihedral features that were actually selected by the model
dihedral_info = []

# Get coefficients and feature indices from the trained model
coef = clf.coef_[0]
model_features = list(zip(selected_idx_final, coef))

# Check each selected feature to see if it's a dihedral feature
for original_idx, coefficient in model_features:
    try:
        # Get feature description
        ftype_desc, atoms_desc, atom_idx = get_feature_atoms(original_idx, feature_info, atom_indices_info, s1_traj[0])
        
        # Check if this is a dihedral feature
        if any(keyword in ftype_desc.lower() for keyword in ['dihedral', 'phi', 'psi', 'chi']):
            # Extract residue number
            words = atoms_desc.split()
            res_num = None
            for word in words:
                if len(word) >= 4 and word[:3].isupper() and word[3:].isdigit():
                    res_num = int(word[3:])
                    break
            
            if res_num is not None:
                # Determine the specific dihedral type
                dihedral_type = ftype_desc  # This will be like 'phi_sin', 'chi1_cos', etc.
                
                dihedral_info.append({
                    'feature_idx': original_idx,
                    'residue': res_num,
                    'type': dihedral_type,
                    'description': atoms_desc,
                    'coefficient': coefficient,
                    'abs_coefficient': abs(coefficient)
                })
                
    except Exception as e:
        continue

print(f"Found {len(dihedral_info)} dihedral features in the selected model features")

# Create separate plots for phi, psi, and chi angles using model coefficients
if dihedral_info:
    # Separate data by angle type
    phi_data = [d for d in dihedral_info if 'phi' in d['type']]
    psi_data = [d for d in dihedral_info if 'psi' in d['type']]
    chi_data = [d for d in dihedral_info if 'chi' in d['type']]
    
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 12))
    
    # Plot 1: Phi angles
    if phi_data:
        phi_residues = [d['residue'] for d in phi_data]
        phi_coeffs = [d['coefficient'] for d in phi_data]
        phi_types = [d['type'] for d in phi_data]
        
        # Color by sin/cos
        for ptype in ['phi_sin', 'phi_cos']:
            mask = np.array(phi_types) == ptype
            if np.any(mask):
                color = 'red' if 'sin' in ptype else 'blue'
                marker = 'o' if 'sin' in ptype else 's'
                ax1.scatter(np.array(phi_residues)[mask], np.array(phi_coeffs)[mask], 
                           c=color, marker=marker, label=ptype, alpha=0.7, s=50)
        
        ax1.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax1.set_ylabel('Logistic Regression Coefficient')
        ax1.set_title('Phi (φ) Dihedral Angles - Selected Features Only')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
    
    # Plot 2: Psi angles
    if psi_data:
        psi_residues = [d['residue'] for d in psi_data]
        psi_coeffs = [d['coefficient'] for d in psi_data]
        psi_types = [d['type'] for d in psi_data]
        
        # Color by sin/cos
        for ptype in ['psi_sin', 'psi_cos']:
            mask = np.array(psi_types) == ptype
            if np.any(mask):
                color = 'red' if 'sin' in ptype else 'blue'
                marker = 'o' if 'sin' in ptype else 's'
                ax2.scatter(np.array(psi_residues)[mask], np.array(psi_coeffs)[mask], 
                           c=color, marker=marker, label=ptype, alpha=0.7, s=50)
        
        ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax2.set_ylabel('Logistic Regression Coefficient')
        ax2.set_title('Psi (ψ) Dihedral Angles - Selected Features Only')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
    
    # Plot 3: Chi angles (sidechain)
    if chi_data:
        chi_residues = [d['residue'] for d in chi_data]
        chi_coeffs = [d['coefficient'] for d in chi_data]
        chi_types = [d['type'] for d in chi_data]
        
        # Color by chi number and sin/cos
        chi_colors = {'chi1': 'green', 'chi2': 'orange', 'chi3': 'purple', 'chi4': 'brown'}
        for chi_type in set(chi_types):
            mask = np.array(chi_types) == chi_type
            if np.any(mask):
                base_chi = chi_type.split('_')[0]  # chi1, chi2, etc.
                color = chi_colors.get(base_chi, 'gray')
                marker = 'o' if 'sin' in chi_type else 's'
                ax3.scatter(np.array(chi_residues)[mask], np.array(chi_coeffs)[mask], 
                           c=color, marker=marker, label=chi_type, alpha=0.7, s=50)
        
        ax3.axhline(y=0, color='black', linestyle='--', alpha=0.5)
        ax3.set_xlabel('Residue Number')
        ax3.set_ylabel('Logistic Regression Coefficient')
        ax3.set_title('Chi (χ) Sidechain Dihedral Angles - Selected Features Only')
        ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        ax3.grid(True, alpha=0.3)
    
    # Set x-axis ticks with interval of 10 for all subplots
    # Get the complete protein residue range
    protein_residues = list(s1_traj[0].topology.residues)
    first_res_id = int(str(protein_residues[0])[3:])
    last_res_id = int(str(protein_residues[-1])[3:])
    
    x_ticks = range(int(first_res_id//10)*10, int(last_res_id//10)*10 + 20, 10)
    
    for ax in [ax1, ax2, ax3]:
        ax.set_xticks(x_ticks)
        ax.set_xlim(first_res_id, last_res_id)
    
    plt.tight_layout()
    plt.savefig('abl_logistic_regression_dihedrals_selected.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Summary statistics for each type
    print(f"Model-selected dihedral features breakdown:")
    print(f"  Phi angles: {len(phi_data)} features")
    print(f"  Psi angles: {len(psi_data)} features") 
    print(f"  Chi angles: {len(chi_data)} features")
    print(f"  Total dihedral features in model: {len(dihedral_info)} out of {len(selected_idx_final)} total features")
    
    # Show top discriminative features for each type
    for angle_type, data, name in [('phi', phi_data, 'Phi'), ('psi', psi_data, 'Psi'), ('chi', chi_data, 'Chi')]:
        if data:
            sorted_data = sorted(data, key=lambda x: abs(x['coefficient']), reverse=True)
            print(f"\nTop {min(3, len(sorted_data))} most important {name} features (by |coefficient|):")
            for i, d in enumerate(sorted_data[:3]):
                direction = "favors S2" if d['coefficient'] > 0 else "favors S1"
                print(f"  {i+1}. Res{d['residue']} {d['type']}: {d['coefficient']:6.3f} ({direction})")
else:
    print("No dihedral features were selected by the logistic regression model.")

RandomForest

---

In [ ]:

# Random Forest pipeline - similar structure but without sparse selection
# (Random Forest has built-in feature selection via importance)
rf_pipe = Pipeline([
    # scale features (helpful for distance features even though RF is tree-based)
    ("scaler", StandardScaler(with_mean=True)),
    # per-fold univariate shrinkage (same as logistic regression)
    ("anova", SelectKBest(score_func=f_classif, k=1000)),
    # Random Forest classifier with built-in feature importance
    ("rf", RandomForestClassifier(random_state=42, n_jobs=-1))
])

rf_param_grid = {
    # how aggressively to pre-shrink with ANOVA
    "anova__k": [300, 500, 800, 1000],
    # Random Forest specific parameters
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [10, 15, 20, None],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4]
}

print("Training Random Forest with GridSearchCV...")
rf_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rf_gs = GridSearchCV(rf_pipe, param_grid=rf_param_grid, cv=rf_cv, scoring="roc_auc", n_jobs=-1)
rf_gs.fit(X, y)

print("Random Forest Best CV AUC:", rf_gs.best_score_)
print("Random Forest Best params:", rf_gs.best_params_)

In [ ]:
# Extract Random Forest components
rf_best_pipe = rf_gs.best_estimator_
rf_anova = rf_best_pipe.named_steps["anova"]
rf_classifier = rf_best_pipe.named_steps["rf"]

# Get ANOVA-selected features for Random Forest
rf_mask_anova = rf_anova.get_support()
rf_selected_idx_anova = selected_idx[rf_mask_anova]

print(f"Random Forest selected {len(rf_selected_idx_anova)} features after ANOVA")
print(f"Random Forest feature importances shape: {rf_classifier.feature_importances_.shape}")

In [ ]:
# Write Random Forest feature importance analysis to file
rf_output_file = Path("egfr_random_forest_features.txt")

with open(rf_output_file, 'w') as f:
    f.write("Random Forest Feature Importance Analysis\n")
    f.write("=" * 60 + "\n\n")
    
    # Get Random Forest feature importances and pair with feature indices
    rf_importances = rf_classifier.feature_importances_
    rf_feature_importance = list(zip(rf_selected_idx_anova, rf_importances))
    
    # Sort by importance (descending)
    rf_feature_importance.sort(key=lambda x: x[1], reverse=True)
    
    f.write(f"Top {len(rf_feature_importance)} features ranked by Random Forest importance:\n")
    f.write("-" * 60 + "\n")
    
    for i, (original_idx, importance) in enumerate(rf_feature_importance):
        ftype, atoms_desc, atom_idx = get_feature_atoms(original_idx, feature_info, atom_indices_info, s1_traj[0])
        f.write(f"{i+1:3d}. Feature {original_idx:4d} (importance={importance:8.6f}): {ftype:12s} - {atoms_desc}\n")
    
    f.write(f"\n" + "=" * 60 + "\n")
    f.write(f"RANDOM FOREST SUMMARY\n")
    f.write(f"=" * 60 + "\n")
    f.write(f"Best CV AUC: {rf_gs.best_score_:.4f}\n")
    f.write(f"Best parameters: {rf_gs.best_params_}\n")
    f.write(f"Total features used: {len(rf_selected_idx_anova)}\n\n")
    
    # Add feature type breakdown
    f.write(f"FEATURE TYPE BREAKDOWN (Random Forest)\n")
    f.write(f"=" * 60 + "\n")
    rf_feature_types = {}
    for original_idx, importance in rf_feature_importance:
        ftype, atoms_desc, atom_idx = get_feature_atoms(original_idx, feature_info, atom_indices_info, s1_traj[0])
        if ftype not in rf_feature_types:
            rf_feature_types[ftype] = []
        rf_feature_types[ftype].append((original_idx, importance, atoms_desc))
    
    for ftype, features in rf_feature_types.items():
        avg_importance = np.mean([imp for _, imp, _ in features])
        f.write(f"\n{ftype.upper()} ({len(features)} features, avg importance: {avg_importance:.6f}):\n")
        f.write("-" * 50 + "\n")
        for original_idx, importance, atoms_desc in features[:10]:  # Show top 10 per type
            f.write(f"  Feature {original_idx:4d} (importance={importance:8.6f}): {atoms_desc}\n")
        if len(features) > 10:
            f.write(f"  ... and {len(features) - 10} more\n")

print(f"Random Forest analysis written to: {rf_output_file.absolute()}")

In [ ]:
coefficient_matrix = np.zeros([len(list(s1_traj[0].topology.residues)), len(list(s1_traj[0].topology.residues))])
first_res_id = int(str(list(s1_traj[0].topology.residues)[0])[3:])
for i, (original_idx, coefficient) in enumerate(rf_feature_importance):
    ftype, atoms_desc, atom_idx = get_feature_atoms(original_idx, feature_info, atom_indices_info, s1_traj[0])
    if 'ca_distances' in ftype:
        atom_id_0 = int(atoms_desc.split(' - ')[0].split()[1][3:]) - first_res_id
        atom_id_1 = int(atoms_desc.split(' - ')[1].split()[1][3:]) - first_res_id

        coefficient_matrix[atom_id_0, atom_id_1] = coefficient
        coefficient_matrix[atom_id_1, atom_id_0] = coefficient

In [ ]:
# Plot coefficient matrix as a 2D heatmap
fig, ax = plt.subplots(figsize=(12, 10))

# Create heatmap with diverging blue-white-red colormap centered at 0
# Calculate symmetric range around 0 for proper centering
vmax = max(abs(coefficient_matrix.min()), abs(coefficient_matrix.max()))
im = ax.imshow(coefficient_matrix, cmap='bwr', aspect='auto', interpolation='nearest', 
               vmin=-vmax, vmax=vmax)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.8)

# Set labels and title
ax.set_xlabel('Residue Index')
ax.set_ylabel('Residue Index')
ax.set_title('CA Distance Features\n(Blue: favors S1/bottleneck, Red: favors S2/post-bottleneck)')

# Add grid for better readability
ax.grid(True, alpha=0.3)

# Set ticks at regular intervals
n_residues = coefficient_matrix.shape[0]
tick_interval = max(1, n_residues // 20)  # Show ~20 ticks
ticks = range(0, n_residues, tick_interval)
ax.set_xticks(ticks)
ax.set_yticks(ticks)

# Add residue numbers (accounting for first_res_id offset)
tick_labels = [str(first_res_id + t) for t in ticks]
ax.set_xticklabels(tick_labels)
ax.set_yticklabels(tick_labels)

plt.tight_layout()
plt.savefig('egfr_random_forest_importance.png', dpi=300)
plt.show()

# Print some statistics about the coefficient matrix
print(f"Matrix shape: {coefficient_matrix.shape}")
print(f"Non-zero elements: {np.count_nonzero(coefficient_matrix)}")
print(f"Min coefficient: {coefficient_matrix.min():.4f}")
print(f"Max coefficient: {coefficient_matrix.max():.4f}")
print(f"Range: {coefficient_matrix.max() - coefficient_matrix.min():.4f}")
print(f"First residue ID: {first_res_id}")
print(f"Last residue ID: {first_res_id + coefficient_matrix.shape[0] - 1}")

In [ ]:
# Train Random Forest if not already done
if 'rf_gs' not in locals():
    print("Random Forest model not found. Please run the Random Forest training cells first.")
else:
    # Extract Random Forest components
    rf_best_pipe = rf_gs.best_estimator_
    rf_anova = rf_best_pipe.named_steps["anova"]
    rf_classifier = rf_best_pipe.named_steps["rf"]
    
    # Get ANOVA-selected features and their importance
    rf_mask_anova = rf_anova.get_support()
    rf_selected_idx_anova = selected_idx[rf_mask_anova]
    rf_importances = rf_classifier.feature_importances_
    rf_feature_importance = list(zip(rf_selected_idx_anova, rf_importances))
    
    print(f"Random Forest selected {len(rf_selected_idx_anova)} features")
    
    # Identify dihedral feature types
    dihedral_types = [k for k in feature_info.keys() 
                      if any(keyword in k.lower() for keyword in ['dihedral', 'phi', 'psi', 'chi'])]
    
    print(f"Found {len(dihedral_types)} dihedral feature types: {', '.join(dihedral_types)}")
    
    # Collect dihedral features and their Random Forest importance
    dihedral_info = []
    
    for original_idx, rf_importance in rf_feature_importance:
        # Check if this feature is a dihedral feature
        for ftype_key in dihedral_types:
            start_idx, end_idx = feature_info[ftype_key]
            if start_idx <= original_idx < end_idx:
                # This is a dihedral feature
                try:
                    ftype_desc, atoms_desc, atom_idx = get_feature_atoms(original_idx, feature_info, atom_indices_info, s1_traj[0])
                    
                    # Extract residue number
                    words = atoms_desc.split()
                    res_num = None
                    for word in words:
                        if len(word) >= 4 and word[:3].isupper() and word[3:].isdigit():
                            res_num = int(word[3:])
                            break
                    
                    if res_num is not None:
                        dihedral_info.append({
                            'feature_idx': original_idx,
                            'residue': res_num,
                            'type': ftype_key,
                            'description': atoms_desc,
                            'rf_importance': rf_importance
                        })
                        
                except (ValueError, IndexError):
                    continue
                break
    
    # Create separate plots for phi, psi, and chi angles using Random Forest importance
    if dihedral_info:
        # Separate data by angle type
        phi_data = [d for d in dihedral_info if 'phi' in d['type']]
        psi_data = [d for d in dihedral_info if 'psi' in d['type']]
        chi_data = [d for d in dihedral_info if 'chi' in d['type']]
        
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 12))
        
        # Plot 1: Phi angles
        if phi_data:
            phi_residues = [d['residue'] for d in phi_data]
            phi_importance = [d['rf_importance'] for d in phi_data]
            phi_types = [d['type'] for d in phi_data]
            
            # Color by sin/cos
            for ptype in ['phi_sin', 'phi_cos']:
                mask = np.array(phi_types) == ptype
                if np.any(mask):
                    color = 'red' if 'sin' in ptype else 'blue'
                    marker = 'o' if 'sin' in ptype else 's'
                    ax1.scatter(np.array(phi_residues)[mask], np.array(phi_importance)[mask], 
                               c=color, marker=marker, label=ptype, alpha=0.7, s=50)
            
            ax1.set_ylabel('Random Forest Importance')
            ax1.set_title('Phi (φ) Dihedral Angles')
            ax1.legend()
            ax1.grid(True, alpha=0.3)
        
        # Plot 2: Psi angles
        if psi_data:
            psi_residues = [d['residue'] for d in psi_data]
            psi_importance = [d['rf_importance'] for d in psi_data]
            psi_types = [d['type'] for d in psi_data]
            
            # Color by sin/cos
            for ptype in ['psi_sin', 'psi_cos']:
                mask = np.array(psi_types) == ptype
                if np.any(mask):
                    color = 'red' if 'sin' in ptype else 'blue'
                    marker = 'o' if 'sin' in ptype else 's'
                    ax2.scatter(np.array(psi_residues)[mask], np.array(psi_importance)[mask], 
                               c=color, marker=marker, label=ptype, alpha=0.7, s=50)
            
            ax2.set_ylabel('Random Forest Importance')
            ax2.set_title('Psi (ψ) Dihedral Angles')
            ax2.legend()
            ax2.grid(True, alpha=0.3)
        
        # Plot 3: Chi angles (sidechain)
        if chi_data:
            chi_residues = [d['residue'] for d in chi_data]
            chi_importance = [d['rf_importance'] for d in chi_data]
            chi_types = [d['type'] for d in chi_data]
            
            # Color by chi number and sin/cos
            chi_colors = {'chi1': 'green', 'chi2': 'orange', 'chi3': 'purple', 'chi4': 'brown'}
            for chi_type in set(chi_types):
                mask = np.array(chi_types) == chi_type
                if np.any(mask):
                    base_chi = chi_type.split('_')[0]  # chi1, chi2, etc.
                    color = chi_colors.get(base_chi, 'gray')
                    marker = 'o' if 'sin' in chi_type else 's'
                    ax3.scatter(np.array(chi_residues)[mask], np.array(chi_importance)[mask], 
                               c=color, marker=marker, label=chi_type, alpha=0.7, s=50)
            
            ax3.set_xlabel('Residue Number')
            ax3.set_ylabel('Random Forest Importance')
            ax3.set_title('Chi (χ) Sidechain Dihedral Angles')
            ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
            ax3.grid(True, alpha=0.3)
        
        # Set x-axis ticks with interval of 10 for all subplots
        all_residues = [d['residue'] for d in dihedral_info]
        if all_residues:
            min_res = min(all_residues)
            max_res = max(all_residues)
            x_ticks = range(int(min_res//10)*10, int(max_res//10)*10 + 20, 10)
            
            for ax in [ax1, ax2, ax3]:
                ax.set_xticks(x_ticks)
                ax.set_xlim(min_res - 10, max_res + 10)
        
        plt.tight_layout()
        plt.savefig('egfr_random_forest_dihedrals.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Summary statistics for each type
        print(f"Analyzed {len(dihedral_info)} total dihedral features:")
        print(f"  Phi angles: {len(phi_data)} features")
        print(f"  Psi angles: {len(psi_data)} features") 
        print(f"  Chi angles: {len(chi_data)} features")
        
        # Show top discriminative features for each type
        for angle_type, data, name in [('phi', phi_data, 'Phi'), ('psi', psi_data, 'Psi'), ('chi', chi_data, 'Chi')]:
            if data:
                sorted_data = sorted(data, key=lambda x: x['rf_importance'], reverse=True)
                print(f"\nTop 3 most important {name} features (Random Forest):")
                for i, d in enumerate(sorted_data[:3]):
                    print(f"  {i+1}. Res{d['residue']} {d['type']}: {d['rf_importance']:6.4f}")